# LegalGPT: train the local models (3B everyday, 7B heavy) on one Colab GPU session

**Before you start**
1. Put `legalgpt_colab.zip` in Google Drive under `MyDrive/legalgpt/`.
2. **Runtime → Change runtime type → L4 GPU** (A100 also works). Not T4: it is about 3× slower, so it
   costs about the same number of units but takes far longer.
3. **Runtime → Run all**, and leave the tab open.

**What it does**, in about 4-5 hours on an L4 (roughly 20-25 compute units; check
*View resources* for the live rate):
1. Trains the **3B** (2 passes), then benchmarks it on the 150 questions.
2. Trains the **7B** (2 passes, the same recipe), benchmarks it, and converts its adapter for
   llama.cpp.
3. **Disconnects the GPU**, so no units are spent after the work is done.

Everything is saved to Drive as it goes. If the session drops, run all again: finished steps are
skipped, and training resumes from its last checkpoint.

In [ ]:
MODELS = ["3B", "7B"]      # run both; e.g. ["7B"] to do only one
WITH_JUDGMENTS = False     # True only if the zip was made with --with-judgments
TAG = "l4"                 # new folders; earlier partial runs (v0.7-3b etc.) are left untouched

BASE = {"3B": "Qwen/Qwen2.5-3B-Instruct", "7B": "Qwen/Qwen2.5-7B-Instruct"}
DRIVE = "/content/drive/MyDrive/legalgpt"
REPO = "/content/legaltech"
run_name = lambda size: f"v0.7-{size.lower()}-{TAG}"
print({s: run_name(s) for s in MODELS})

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)
if "T4" in gpu:
    print("WARNING: this is a T4. Change the runtime to L4 (Runtime -> Change runtime type) for a ~3x faster run.")

In [ ]:
import os, zipfile, time
if not os.path.exists(REPO):
    zipfile.ZipFile(f"{DRIVE}/legalgpt_colab.zip").extractall(REPO)
os.makedirs(f"{DRIVE}/bench", exist_ok=True)
os.makedirs(f"{REPO}/data/reports/bench", exist_ok=True)
link = f"{REPO}/data/reports/bench/baselines"
if not os.path.exists(link):
    os.symlink(f"{DRIVE}/bench", link)   # benchmark answers go straight to Drive
!pip install -q -r {REPO}/backend/scripts/colab/requirements-colab.txt
!pip uninstall -y -q torchao   # Colab's preinstalled torchao 0.10 makes PEFT refuse to load adapters
import torch, transformers
print("torch", torch.__version__, "| bf16", torch.cuda.is_bf16_supported(), "| transformers", transformers.__version__)

## Train, benchmark and convert (both models)
- **Training recipe:** the same as the laptop's v0.7, 2 passes. On a 24 GB GPU it reads 4 examples
  at a time with 2 accumulation steps, the same effective batch of 8 as 1 × 8 on the laptop, only
  faster.
- **Benchmark:** fully offline. The local model rewrites each question itself.

In [ ]:
import shlex

def sh(cmd, env=None):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    # progress bars update once a minute instead of many times a second
    env = {**os.environ, "TQDM_MININTERVAL": "60", "PYTHONUNBUFFERED": "1", **(env or {})}
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd[:120]}")

judg = "" if WITH_JUDGMENTS else "--no-judgments"
big_gpu = torch.cuda.get_device_properties(0).total_memory > 20e9
batch_plans = [(4, 2), (2, 4), (1, 8)] if big_gpu else [(1, 8)]   # effective batch 8 in every plan
t0 = time.time()
hours = lambda: f"{(time.time() - t0) / 3600:.1f} h so far"
for size in MODELS:
    run, base = run_name(size), BASE[size]
    out = f"{DRIVE}/lora/{run}"
    # 1. train (skipped if this model finished training in an earlier session)
    if not os.path.exists(f"{out}/adapter_model.safetensors"):
        for bs, ga in batch_plans:  # out of GPU memory -> retry with fewer examples per step
            print(f"\n=== training {size} (batch {bs} x {ga}) -> {out}  ({hours()})")
            try:
                sh(f"python backend/scripts/train_lora.py --base-model {base} --version v0.2 {judg} "
                   f"--ipc-qa --no-case-refusals --scenarios --scenario-refusals 0.3 --rewrites --epochs 2 "
                   f"--batch-size {bs} --grad-accum {ga} --save-steps 20 --out {shlex.quote(out)}")
                break
            except RuntimeError as e:
                print(e, "- retrying with a smaller batch" if (bs, ga) != batch_plans[-1] else "")
                torch.cuda.empty_cache()
        else:
            raise RuntimeError(f"training {size} failed with every batch size")
    # 2. benchmark (resumes: answers already in Drive are skipped)
    print(f"\n=== benchmark {size}  ({hours()})")
    sh(f"python backend/scripts/run_baselines.py run --rewrite --suffix _{run} --systems legalgpt-lora",
       env={"ADAPTER_PATH": out, "LOCAL_LLM_DTYPE": "auto"})
    # 3. 7B only: the adapter as GGUF for llama.cpp on the laptop
    if size == "7B" and not os.path.exists(f"{out}.gguf"):
        print(f"\n=== converting the 7B adapter to GGUF  ({hours()})")
        if not os.path.exists("/content/llama.cpp"):
            sh("git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp")
        sh("pip install -q /content/llama.cpp/gguf-py sentencepiece")
        from huggingface_hub import snapshot_download
        base_cfg = snapshot_download(base, allow_patterns=["*.json", "tokenizer*", "*.model"])
        sh(f"python /content/llama.cpp/convert_lora_to_gguf.py --base {base_cfg} --outtype f16 "
           f"--outfile {shlex.quote(out + '.gguf')} {shlex.quote(out)}")
print(f"\nall done in {(time.time() - t0) / 3600:.1f} h")

In [ ]:
sh("python backend/scripts/run_baselines.py score")

In [ ]:
# stop the GPU so no more compute units are used (the files are all in Drive)
from google.colab import runtime
runtime.unassign()

## Bring it to the laptop
From Google Drive, `MyDrive/legalgpt/`:
- **3B:** the folder `lora/v0.7-3b-l4`, without its `_trainer` subfolder. Put it in
  `data/models/lora/` and set `ADAPTER_PATH=data/models/lora/v0.7-3b-l4` in `.env`.
- **7B:** `lora/v0.7-7b-l4.gguf`. Put it in `data/models/gguf/`; see `backend/scripts/colab/README.md`.
- **Results:** `bench/legalgpt-lora_v0.7-*-l4.jsonl`, into `data/reports/bench/baselines/`.